In [ ]:
print("🚀 Production Readiness Assessment")
print("="*50)

# Define production readiness criteria
readiness_criteria = {
    'workflow_completion': len(successful_fields) == len(target_fields),
    'file_generation': all(os.path.exists(result['output_file']) for result in workflow_results.values() if result['success']),
    'data_validation': all(result['samples'] > 0 for result in workflow_results.values() if result['success']),
    'umls_integration': all(pd.read_csv(result['output_file'])['CUI'].notna().all() 
                           for result in workflow_results.values() 
                           if result['success'] and os.path.exists(result['output_file'])),
    'minimum_coverage': len(successful_fields) >= 2  # At least 2 fields must be processed
}

# Evaluate each criterion
print(f"\n📋 Production Readiness Checklist:")
readiness_score = 0
total_criteria = len(readiness_criteria)

for criterion, passed in readiness_criteria.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    criterion_display = criterion.replace('_', ' ').title()
    print(f"   • {criterion_display}: {status}")
    if passed:
        readiness_score += 1

# Calculate readiness percentage
readiness_percentage = (readiness_score / total_criteria) * 100

print(f"\n🎯 Production Readiness Score: {readiness_score}/{total_criteria} ({readiness_percentage:.1f}%)")

# Provide recommendation
if readiness_percentage >= 90:
    print("\n🟢 PRODUCTION READY")
    print("   ✅ All critical criteria met")
    print("   🚀 Workflow is ready for production deployment")
    print("   📈 High confidence in data processing pipeline")
elif readiness_percentage >= 70:
    print("\n🟡 MOSTLY READY")
    print("   ⚠️ Minor issues detected")
    print("   🔧 Review failed criteria and address issues")
    print("   📊 Consider additional testing before full deployment")
else:
    print("\n🔴 NOT READY")
    print("   ❌ Critical issues must be resolved")
    print("   🛠️ Address all failed criteria before production use")
    print("   🧪 Additional development and testing required")

# Provide specific recommendations
failed_criteria = [criterion for criterion, passed in readiness_criteria.items() if not passed]

if failed_criteria:
    print(f"\n📝 Recommendations for improvement:")
    for criterion in failed_criteria:
        if criterion == 'workflow_completion':
            print("   • Ensure all target fields can be processed successfully")
        elif criterion == 'file_generation':
            print("   • Verify output file generation and path accessibility")
        elif criterion == 'data_validation':
            print("   • Check for empty or invalid output files")
        elif criterion == 'umls_integration':
            print("   • Validate UMLS matching and CUI assignment")
        elif criterion == 'minimum_coverage':
            print("   • Ensure at least 2 fields are processed successfully")

print(f"\n📊 Assessment completed at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 All output files saved to: {output_dir}")
print(f"📋 Ready for downstream analysis and validation")

## 🚀 Production Readiness Assessment

In [ ]:
print("📈 Workflow Statistics Dashboard")
print("="*50)

if successful_fields:
    # Create comprehensive statistics summary
    stats_summary = []
    
    for field in successful_fields:
        result = workflow_results[field]
        
        # Calculate percentages and ratios
        original_samples = consolidated_df[field].notna().sum() if field in consolidated_df.columns else 0
        processed_samples = result['samples']
        processing_rate = (processed_samples / original_samples * 100) if original_samples > 0 else 0
        
        stats_summary.append({
            'Field': field.replace('_', ' ').title(),
            'Original Samples': f"{original_samples:,}",
            'UMLS Matched': f"{processed_samples:,}",
            'Match Rate': f"{processing_rate:.1f}%",
            'File Size (KB)': f"{result['file_size']/1024:.1f}",
            'Output File': os.path.basename(result['output_file'])
        })
    
    stats_df = pd.DataFrame(stats_summary)
    print("\n📊 Comprehensive Processing Statistics:")
    display(stats_df)
    
    # Overall workflow metrics
    total_original = sum(consolidated_df[field].notna().sum() for field in successful_fields if field in consolidated_df.columns)
    total_processed = sum(result['samples'] for result in workflow_results.values() if result['success'])
    total_file_size = sum(result['file_size'] for result in workflow_results.values() if result['success'])
    
    print(f"\n🎯 Overall Workflow Metrics:")
    print(f"   • Total samples with target fields: {total_original:,}")
    print(f"   • Total UMLS-matched samples: {total_processed:,}")
    print(f"   • Overall matching efficiency: {(total_processed/total_original*100):.1f}%")
    print(f"   • Total output size: {total_file_size/1024:.1f} KB ({total_file_size/1024/1024:.2f} MB)")
    print(f"   • Average file size: {(total_file_size/len(successful_fields))/1024:.1f} KB per field")
    
    # Data density analysis
    if consolidated_df is not None:
        print(f"\n📋 Data Density Analysis:")
        for field in successful_fields:
            if field in consolidated_df.columns:
                unique_values = consolidated_df[field].nunique()
                samples_per_value = consolidated_df[field].notna().sum() / unique_values if unique_values > 0 else 0
                
                print(f"   • {field.replace('_', ' ').title()}:")
                print(f"     - Unique values: {unique_values}")
                print(f"     - Average samples per value: {samples_per_value:.1f}")

print(f"\n✅ Statistics dashboard completed")

## 📈 Workflow Statistics Dashboard

# 🔄 First-Pass Workflow Complete Test

This notebook walks through the complete first-pass table generation workflow, step by step.

## Workflow steps

```
META files (GSE*.txt)
     ↓
1. Load and merge
     ↓
2. Filter (Homo sapiens, TRANSCRIPTOMIC)
     ↓
3. Consolidate fields (field_consolidation.py)
   - build disease_state_modified
   - merge the sex fields
     ↓
4. UMLS matching (umls_readers.py)
   - CSV-backed lookup
   - apply SAB priority order
     ↓
5. Keep only rows that got a CUI
     ↓
6. Write CSV output
   - HS_cell_type_1st_pass_meta_table.csv
   - HS_tissue_1st_pass_meta_table.csv
   - HS_disease_1st_pass_meta_table.csv
```

## 🔧 Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Import required modules
import genoar_analysis as ga
from genoar_analysis.io.umls_readers import UMLSCSVReader
from genoar_analysis.pipelines.first_pass_pipeline import FirstPassPipeline

print("✅ Setup complete")

## 🗂️ Directory setup and checks

In [ ]:
# Directory paths, by the same convention as conftest.py in this directory:
# the environment variable when it is set, otherwise a default relative to
# the repository root. The notebook is run from its own directory, which is
# what the sys.path line in the setup cell already assumes.
REPO_ROOT = Path.cwd().parents[1]

meta_dir = os.environ.get("GENOAR_META_DIR", str(REPO_ROOT / "sample_crawl_output" / "META"))
umls_dir = os.environ.get("GENOAR_UMLS_DIR", str(REPO_ROOT / "all_query_results"))
output_dir = os.environ.get("GENOAR_OUTPUT_DIR", str(REPO_ROOT / "test_first_pass_output"))

# Create output directory
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory: {output_dir}")

# Check input directories
print("\n🔍 Directory Check:")
meta_exists = os.path.exists(meta_dir)
umls_exists = os.path.exists(umls_dir)
print(f"META directory: {'✅ Found' if meta_exists else '❌ Not found'} - {meta_dir}")
print(f"UMLS directory: {'✅ Found' if umls_exists else '❌ Not found'} - {umls_dir}")

# Count files
if meta_exists:
    meta_files = list(Path(meta_dir).glob("*_meta.txt"))
    print(f"📊 Found {len(meta_files)} META files")
    if len(meta_files) > 0:
        print("📋 Sample files:")
        for i, file in enumerate(meta_files[:5]):
            print(f"  • {file.name}")
        if len(meta_files) > 5:
            print(f"  ... and {len(meta_files) - 5} more files")
else:
    meta_files = []

## 📊 Step 1: Load and merge

In [ ]:
print("📊 Step 1: Data Loading and Merging")
print("="*50)

if not meta_exists or len(meta_files) == 0:
    print("❌ No META files found - cannot proceed with data loading")
    combined_df = None
else:
    try:
        # Initialize pipeline
        pipeline = FirstPassPipeline(meta_dir, umls_dir)
        print("✅ Pipeline initialized")
        
        # Load and preprocess data
        combined_df = pipeline.load_and_preprocess_data()
        
        print(f"✅ Data loaded successfully")
        print(f"📊 Combined data shape: {combined_df.shape}")
        print(f"📊 Unique series: {combined_df['Series'].nunique()}")
        print(f"📊 Total samples: {len(combined_df)}")
        
    except Exception as e:
        print(f"❌ Data loading failed: {e}")
        combined_df = None

In [ ]:
# Display data information
if combined_df is not None:
    # Show column information
    important_cols = ['tissue', 'cell_type', 'disease_state', 'disease', 'Organism', 'LibrarySource']
    available_important = [col for col in important_cols if col in combined_df.columns]
    
    print(f"\n📋 Available important columns: {available_important}")
    print(f"📋 All columns ({len(combined_df.columns)}): {list(combined_df.columns)[:10]}{'...' if len(combined_df.columns) > 10 else ''}")
    
    # Show sample data
    print("\n📋 Sample data (first 3 rows):")
    display_cols = ['Series', 'Run'] + available_important
    display_cols = [col for col in display_cols if col in combined_df.columns]
    display(combined_df[display_cols].head(3))
    
    print("\n🎉 Step 1 COMPLETED")
else:
    print("❌ Step 1 FAILED - No data to proceed with")

## 🔧 Step 2: Filter (Homo sapiens, TRANSCRIPTOMIC)

In [ ]:
print("🔧 Step 2: Filtering (Homo sapiens, TRANSCRIPTOMIC)")
print("="*50)

if combined_df is None:
    print("❌ No data from Step 1 - cannot proceed")
    filtered_df = None
else:
    filtered_df = combined_df.copy()
    original_count = len(filtered_df)
    print(f"📊 Starting samples: {original_count:,}")
    
    # Filter by Organism
    if 'Organism' in filtered_df.columns:
        organism_counts = filtered_df['Organism'].value_counts()
        print(f"\n🦠 Organism distribution:")
        for org, count in organism_counts.head().items():
            print(f"  • {org}: {count:,} samples")
        
        before_organism = len(filtered_df)
        filtered_df = filtered_df[filtered_df['Organism'] == 'Homo sapiens']
        after_organism = len(filtered_df)
        
        print(f"\n✅ Organism filter: {before_organism:,} → {after_organism:,} samples")
        print(f"   Removed: {before_organism - after_organism:,} non-human samples")
    else:
        print("⚠️ No Organism column found - skipping organism filter")

In [ ]:
# Filter by LibrarySource
if filtered_df is not None and 'LibrarySource' in filtered_df.columns:
    library_counts = filtered_df['LibrarySource'].value_counts()
    print(f"\n📚 LibrarySource distribution:")
    for lib, count in library_counts.head().items():
        print(f"  • {lib}: {count:,} samples")
    
    before_library = len(filtered_df)
    filtered_df = filtered_df[filtered_df['LibrarySource'].str.contains('TRANSCRIPTOMIC', na=False)]
    after_library = len(filtered_df)
    
    print(f"\n✅ LibrarySource filter: {before_library:,} → {after_library:,} samples")
    print(f"   Removed: {before_library - after_library:,} non-transcriptomic samples")
elif filtered_df is not None:
    print("⚠️ No LibrarySource column found - skipping library filter")

if filtered_df is not None:
    final_count = len(filtered_df)
    reduction_pct = (original_count - final_count) / original_count * 100
    print(f"\n📊 Final sample count: {final_count:,} ({reduction_pct:.1f}% reduction from original)")
    print("\n🎉 Step 2 COMPLETED")

## 🔀 Step 3: Consolidate fields

In [ ]:
print("🔀 Step 3: Field Consolidation")
print("="*50)

if filtered_df is None:
    print("❌ No data from Step 2 - cannot proceed")
    consolidated_df = None
else:
    try:
        # Use pipeline's consolidate_fields method
        pipeline = FirstPassPipeline(meta_dir, umls_dir)
        consolidated_df = pipeline.consolidate_fields(filtered_df.copy())
        
        print("✅ Field consolidation completed")
        
    except Exception as e:
        print(f"❌ Field consolidation failed: {e}")
        consolidated_df = filtered_df.copy()
        print("⚠️ Using original filtered data without consolidation")

In [ ]:
# Check consolidation results
if consolidated_df is not None:
    # Check for disease_state_modified creation
    if 'disease_state_modified' in consolidated_df.columns:
        disease_count = consolidated_df['disease_state_modified'].notna().sum()
        disease_unique = consolidated_df['disease_state_modified'].nunique()
        print(f"✅ disease_state_modified: {disease_count:,} non-null values, {disease_unique} unique values")
        
        # Show top disease states
        if disease_count > 0:
            top_diseases = consolidated_df['disease_state_modified'].value_counts().head()
            print("📊 Top disease states:")
            for disease, count in top_diseases.items():
                if pd.notna(disease):
                    print(f"  • {disease}: {count:,} samples")
    else:
        print("⚠️ disease_state_modified column not created")
    
    # Check sex field consolidation
    if 'sex' in consolidated_df.columns:
        sex_count = consolidated_df['sex'].notna().sum()
        sex_dist = consolidated_df['sex'].value_counts()
        print(f"\n✅ sex field: {sex_count:,} non-null values")
        print("📊 Sex distribution:")
        for sex, count in sex_dist.items():
            if pd.notna(sex):
                print(f"  • {sex}: {count:,} samples")
    else:
        print("⚠️ sex column not found or not consolidated")
    
    # Show available fields for analysis
    analysis_fields = ['tissue', 'cell_type', 'disease_state_modified', 'sex']
    available_fields = [f for f in analysis_fields if f in consolidated_df.columns]
    print(f"\n📋 Available analysis fields: {available_fields}")
    
    print("\n🎉 Step 3 COMPLETED")

## 🔍 Steps 4-6: UMLS matching and output, per field

In [ ]:
# Initialize results storage
workflow_results = {}
target_fields = ['cell_type', 'tissue', 'disease_state_modified']

print("🔍 Fields to process:")
for field in target_fields:
    if consolidated_df is not None and field in consolidated_df.columns:
        non_null_count = consolidated_df[field].notna().sum()
        unique_count = consolidated_df[field].nunique()
        print(f"  • {field}: {non_null_count:,} samples, {unique_count} unique values")
    else:
        print(f"  • {field}: ❌ Not available")

### 🧬 Cell Type Processing

In [ ]:
print("🧬 Processing Cell Type Field")
print("="*40)

field = 'cell_type'

if consolidated_df is None or field not in consolidated_df.columns:
    print(f"❌ {field} field not available")
    workflow_results[field] = {'success': False, 'error': 'Field not available'}
else:
    try:
        # Use pipeline method
        pipeline = FirstPassPipeline(meta_dir, umls_dir)
        output_filename = f"HS_{field}_1st_pass_meta_table.csv"
        output_path = os.path.join(output_dir, output_filename)
        
        print(f"🔄 Processing {field} with pipeline method...")
        first_pass_df = pipeline.create_first_pass_table(field, output_path=output_path)
        
        if len(first_pass_df) > 0:
            workflow_results[field] = {
                'success': True,
                'samples': len(first_pass_df),
                'output_file': output_path,
                'file_size': os.path.getsize(output_path) if os.path.exists(output_path) else 0
            }
            
            print(f"✅ {field} processing completed:")
            print(f"   📊 Samples: {len(first_pass_df):,}")
            print(f"   📄 File: {output_filename}")
            print(f"   💾 Size: {workflow_results[field]['file_size']:,} bytes")
            
            # Show sample of results
            if len(first_pass_df) > 0:
                print("\n📋 Sample results:")
                display_cols = ['Series', field, 'CUI', 'STR', 'SAB'] if 'CUI' in first_pass_df.columns else ['Series', field]
                display_cols = [col for col in display_cols if col in first_pass_df.columns]
                display(first_pass_df[display_cols].head(3))
        else:
            workflow_results[field] = {'success': False, 'error': 'No data generated'}
            print(f"⚠️ {field} processing completed but no data generated")
            
    except Exception as e:
        workflow_results[field] = {'success': False, 'error': str(e)}
        print(f"❌ {field} processing failed: {e}")

### 🫀 Tissue Processing

In [ ]:
print("🫀 Processing Tissue Field")
print("="*40)

field = 'tissue'

if consolidated_df is None or field not in consolidated_df.columns:
    print(f"❌ {field} field not available")
    workflow_results[field] = {'success': False, 'error': 'Field not available'}
else:
    try:
        pipeline = FirstPassPipeline(meta_dir, umls_dir)
        output_filename = f"HS_{field}_1st_pass_meta_table.csv"
        output_path = os.path.join(output_dir, output_filename)
        
        print(f"🔄 Processing {field} with pipeline method...")
        first_pass_df = pipeline.create_first_pass_table(field, output_path=output_path)
        
        if len(first_pass_df) > 0:
            workflow_results[field] = {
                'success': True,
                'samples': len(first_pass_df),
                'output_file': output_path,
                'file_size': os.path.getsize(output_path) if os.path.exists(output_path) else 0
            }
            
            print(f"✅ {field} processing completed:")
            print(f"   📊 Samples: {len(first_pass_df):,}")
            print(f"   📄 File: {output_filename}")
            print(f"   💾 Size: {workflow_results[field]['file_size']:,} bytes")
            
            # Show sample of results
            if len(first_pass_df) > 0:
                print("\n📋 Sample results:")
                display_cols = ['Series', field, 'CUI', 'STR', 'SAB'] if 'CUI' in first_pass_df.columns else ['Series', field]
                display_cols = [col for col in display_cols if col in first_pass_df.columns]
                display(first_pass_df[display_cols].head(3))
        else:
            workflow_results[field] = {'success': False, 'error': 'No data generated'}
            print(f"⚠️ {field} processing completed but no data generated")
            
    except Exception as e:
        workflow_results[field] = {'success': False, 'error': str(e)}
        print(f"❌ {field} processing failed: {e}")

### 🩺 Disease State Processing

In [ ]:
print("🩺 Processing Disease State Field")
print("="*40)

field = 'disease_state_modified'
output_field = 'disease'  # Output filename uses 'disease'

if consolidated_df is None or field not in consolidated_df.columns:
    print(f"❌ {field} field not available")
    workflow_results[field] = {'success': False, 'error': 'Field not available'}
else:
    try:
        pipeline = FirstPassPipeline(meta_dir, umls_dir)
        output_filename = f"HS_{output_field}_1st_pass_meta_table.csv"
        output_path = os.path.join(output_dir, output_filename)
        
        print(f"🔄 Processing {field} with pipeline method...")
        first_pass_df = pipeline.create_first_pass_table(field, output_path=output_path)
        
        if len(first_pass_df) > 0:
            workflow_results[field] = {
                'success': True,
                'samples': len(first_pass_df),
                'output_file': output_path,
                'file_size': os.path.getsize(output_path) if os.path.exists(output_path) else 0
            }
            
            print(f"✅ {field} processing completed:")
            print(f"   📊 Samples: {len(first_pass_df):,}")
            print(f"   📄 File: {output_filename}")
            print(f"   💾 Size: {workflow_results[field]['file_size']:,} bytes")
            
            # Show sample of results
            if len(first_pass_df) > 0:
                print("\n📋 Sample results:")
                display_cols = ['Series', field, 'CUI', 'STR', 'SAB'] if 'CUI' in first_pass_df.columns else ['Series', field]
                display_cols = [col for col in display_cols if col in first_pass_df.columns]
                display(first_pass_df[display_cols].head(3))
        else:
            workflow_results[field] = {'success': False, 'error': 'No data generated'}
            print(f"⚠️ {field} processing completed but no data generated")
            
    except Exception as e:
        workflow_results[field] = {'success': False, 'error': str(e)}
        print(f"❌ {field} processing failed: {e}")

## 📊 Workflow results summary

In [ ]:
print("="*70)
print("📊 WORKFLOW TEST SUMMARY")
print("="*70)

# Compile results
successful_fields = []
failed_fields = []

results_data = []
for field, result in workflow_results.items():
    field_display = field.replace('_', ' ').replace('state modified', 'state').title()
    
    if result['success']:
        successful_fields.append(field)
        results_data.append({
            'Field': field_display,
            'Status': '✅ SUCCESS',
            'Samples': f"{result['samples']:,}",
            'File Size': f"{result['file_size']:,} bytes",
            'Output File': os.path.basename(result['output_file'])
        })
    else:
        failed_fields.append(field)
        results_data.append({
            'Field': field_display,
            'Status': '❌ FAILED',
            'Samples': '0',
            'File Size': '0 bytes',
            'Output File': f"Error: {result.get('error', 'Unknown')}"
        })

if results_data:
    results_df = pd.DataFrame(results_data)
    display(results_df)
else:
    print("⚠️ No results to display")

In [ ]:
# Summary statistics
total_fields = len(workflow_results)
success_count = len(successful_fields)

print(f"\n📊 Overall Results:")
print(f"• Total fields processed: {total_fields}")
print(f"• Successful: {success_count}")
print(f"• Failed: {total_fields - success_count}")
print(f"• Success rate: {success_count/total_fields*100:.1f}%" if total_fields > 0 else "• Success rate: N/A")

if successful_fields:
    total_samples = sum(workflow_results[field]['samples'] for field in successful_fields)
    total_file_size = sum(workflow_results[field]['file_size'] for field in successful_fields)
    
    print(f"\n📈 Success Metrics:")
    print(f"• Total samples in output files: {total_samples:,}")
    print(f"• Total output file size: {total_file_size:,} bytes ({total_file_size/1024/1024:.2f} MB)")
    
    print(f"\n📁 Generated Files:")
    for field in successful_fields:
        result = workflow_results[field]
        filename = os.path.basename(result['output_file'])
        print(f"• {filename} ({result['samples']:,} samples)")

print(f"\n📁 Output directory: {output_dir}")

## 📝 Workflow validation

In [ ]:
print("📝 Workflow Validation")
print("="*50)

# Check if output files exist and are readable
validation_results = []

for field, result in workflow_results.items():
    if result['success']:
        output_file = result['output_file']
        
        validation = {
            'field': field,
            'file_exists': os.path.exists(output_file),
            'file_readable': False,
            'has_data': False,
            'has_cui_column': False,
            'sample_count': 0
        }
        
        if validation['file_exists']:
            try:
                df = pd.read_csv(output_file)
                validation['file_readable'] = True
                validation['has_data'] = len(df) > 0
                validation['sample_count'] = len(df)
                validation['has_cui_column'] = 'CUI' in df.columns
                
            except Exception as e:
                print(f"⚠️ Could not read {os.path.basename(output_file)}: {e}")
        
        validation_results.append(validation)

# Display validation results
if validation_results:
    validation_data = []
    for val in validation_results:
        field_name = val['field'].replace('_state_modified', '').replace('_', ' ').title()
        validation_data.append({
            'Field': field_name,
            'File Exists': '✅' if val['file_exists'] else '❌',
            'Readable': '✅' if val['file_readable'] else '❌',
            'Has Data': '✅' if val['has_data'] else '❌',
            'Has CUI': '✅' if val['has_cui_column'] else '❌',
            'Samples': val['sample_count']
        })
    
    validation_df = pd.DataFrame(validation_data)
    display(validation_df)
    
    # Overall validation status
    all_valid = all(val['file_exists'] and val['file_readable'] and val['has_data'] and val['has_cui_column'] 
                   for val in validation_results)
    
    if all_valid:
        print("\n🎉 All output files are valid and contain UMLS-matched data!")
    else:
        print("\n⚠️ Some output files have validation issues")
else:
    print("⚠️ No successful results to validate")

## 🎯 Final verdict

In [ ]:
print("🎯 FINAL RESULTS")
print("="*50)

if len(successful_fields) > 0:
    print("✅ WORKFLOW SUCCESS!")
    print("\nThe first-pass workflow has been completed successfully:")
    
    workflow_steps = [
        "1. ✅ META files loaded and merged",
        "2. ✅ Data filtered (Homo sapiens, TRANSCRIPTOMIC)",
        "3. ✅ Fields consolidated (disease_state_modified, sex)",
        "4. ✅ UMLS matching performed",
        "5. ✅ CUI filtering applied",
        "6. ✅ CSV files generated"
    ]
    
    for step in workflow_steps:
        print(f"   {step}")
    
    print(f"\n📊 Generated {len(successful_fields)} first-pass tables:")
    for field in successful_fields:
        result = workflow_results[field]
        filename = os.path.basename(result['output_file'])
        print(f"   • {filename}")
    
    print("\n🚀 Ready for production use!")
    
elif len(workflow_results) > 0:
    print("⚠️ WORKFLOW PARTIAL SUCCESS")
    print("\nSome parts of the workflow completed, but not all fields were processed successfully.")
    print("\nThis may be due to:")
    print("   • Missing UMLS data files")
    print("   • Insufficient data in specific fields")
    print("   • Configuration issues")
    
    if successful_fields:
        print(f"\n✅ Successfully processed: {', '.join(successful_fields)}")
    if failed_fields:
        print(f"❌ Failed to process: {', '.join(failed_fields)}")
        
else:
    print("❌ WORKFLOW FAILED")
    print("\nThe workflow could not be completed. This may be due to:")
    print("   • Missing META files")
    print("   • Missing UMLS data")
    print("   • Configuration or setup issues")
    print("\nPlease check the error messages above and ensure all required files are available.")

print(f"\n📝 Note: This workflow creates UMLS-matched metadata tables without Ground Truth comparison.")
print(f"📁 All output files are saved in: {output_dir}")

In [ ]:
print("🧪 Additional Quality Assurance Checks")
print("="*50)

# Performance metrics
import time
import psutil

process = psutil.Process()
memory_usage = process.memory_info().rss / 1024 / 1024  # MB

print(f"\n⚡ Performance Metrics:")
print(f"   • Current memory usage: {memory_usage:.1f} MB")
print(f"   • Workflow completion time: Available in individual step outputs")

# Data quality checks
if successful_fields and consolidated_df is not None:
    print(f"\n📊 Data Quality Assessment:")
    
    # Check data completeness for each processed field
    for field in successful_fields:
        if field in consolidated_df.columns:
            total_samples = len(consolidated_df)
            non_null_samples = consolidated_df[field].notna().sum()
            completeness = (non_null_samples / total_samples) * 100
            
            # Load the output file to check processing efficiency
            result = workflow_results[field]
            if 'output_file' in result and os.path.exists(result['output_file']):
                output_df = pd.read_csv(result['output_file'])
                processed_samples = len(output_df)
                processing_efficiency = (processed_samples / non_null_samples) * 100 if non_null_samples > 0 else 0
                
                print(f"   • {field}:")
                print(f"     - Data completeness: {completeness:.1f}% ({non_null_samples:,}/{total_samples:,})")
                print(f"     - UMLS matching rate: {processing_efficiency:.1f}% ({processed_samples:,}/{non_null_samples:,})")
                
                # Check for unique CUIs in output
                if 'CUI' in output_df.columns:
                    unique_cuis = output_df['CUI'].nunique()
                    print(f"     - Unique CUIs found: {unique_cuis}")

# File integrity checks
print(f"\n🔍 File Integrity Verification:")
for field, result in workflow_results.items():
    if result['success']:
        output_file = result['output_file']
        if os.path.exists(output_file):
            file_size = os.path.getsize(output_file)
            
            # Read and verify file structure
            try:
                df = pd.read_csv(output_file)
                required_cols = ['Series', field, 'CUI', 'STR']
                missing_cols = [col for col in required_cols if col not in df.columns]
                
                field_display = field.replace('_', ' ').title()
                print(f"   • {field_display}:")
                print(f"     - File size: {file_size:,} bytes")
                print(f"     - Rows: {len(df):,}")
                print(f"     - Columns: {len(df.columns)}")
                
                if missing_cols:
                    print(f"     - ⚠️ Missing columns: {missing_cols}")
                else:
                    print(f"     - ✅ All required columns present")
                
                # Check for data consistency
                null_cuis = df['CUI'].isna().sum()
                if null_cuis > 0:
                    print(f"     - ⚠️ {null_cuis} rows with missing CUI")
                else:
                    print(f"     - ✅ All rows have valid CUI")
                    
            except Exception as e:
                print(f"     - ❌ File validation failed: {e}")

print(f"\n✅ Quality assurance checks completed")